<a href="https://colab.research.google.com/github/siva123h/LLM/blob/main/Experiments_no_6_docx_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ===============================
# STEP 1: Install Required Libraries
# ===============================
!pip -q install transformers sentence-transformers faiss-cpu datasets


# ===============================
# STEP 2: Import Libraries
# ===============================
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import numpy as np
import faiss


# ===============================
# STEP 3: Create Knowledge Base
# ===============================
documents = [
    "Artificial Intelligence is the simulation of human intelligence in machines.",
    "Machine Learning is a subset of AI that allows systems to learn from data.",
    "Deep Learning uses neural networks with many layers.",
    "Retrieval Augmented Generation combines retrieval and language models.",
    "FAISS is a a library developed by Facebook for fast similarity search.",
    "Hugging Face provides thousands of pretrained NLP models.",
    "Natural Language Processing allows computers to understand human language.",
    "Large Language Models are trained using huge amounts of text data."
]


# ===============================
# STEP 4: Load Embedding Model
# ===============================
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# ===============================
# STEP 5: Convert Documents to Embeddings
# ===============================
doc_embeddings = embedding_model.encode(documents)
doc_embeddings = np.array(doc_embeddings)


# ===============================
# STEP 6: Create FAISS Vector Index
# ===============================
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)


# ===============================
# STEP 7: Load Language Model
# ===============================
generator = pipeline(
    "text-generation",
    model="gpt2",
    max_length=120,
    truncation=True
)


# ===============================
# STEP 8: Document Retrieval Function
# ===============================
def retrieve_documents(query, top_k=2):

    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding)

    distances, indices = index.search(query_embedding, top_k)

    retrieved_docs = [documents[i] for i in indices[0]]

    return retrieved_docs


# ===============================
# STEP 9: RAG Answer Generation
# ===============================
def rag_pipeline(query):

    docs = retrieve_documents(query)

    context = " ".join(docs)

    prompt = f"""
Context: {context}

Question: {query}

Answer:
"""

    result = generator(prompt)[0]["generated_text"]

    return result


# ===============================
# STEP 10: Ask a Question
# ===============================
query = input("Enter your question: ")

answer = rag_pipeline(query)

print("\nGenerated Answer:\n")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Enter your question: what is deep learning


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generated Answer:


Context: Deep Learning uses neural networks with many layers. Machine Learning is a subset of AI that allows systems to learn from data.

Question: what is deep learning

Answer:

Deep Learning is the field of computing that is
